# 2DTFIM 1DNQS - (Nx,Ny)=(10,10): Inference (seed 111)

This is part of the work arxiv: 2606.25600 (Two-dimensional Hyperbolic RNN Neural Quantum State). For the purpose of reproducing the results, please check the link to the trained weight files. The saved weight links used in this notebook might not be the same as the ones in the Github repo. 

In [1]:
import sys
import os
sys.path.append('../../utility_tfim')
from lorentz_tfim2d_1drnn_train_loop import *
from poincare_tfim2d_1drnn_train_loop import *
import time

Hypercore Lorentzian module loaded successfully with Geoopt wrappers.


In [2]:
def set_cpu_deterministic(seed=111):
    # 1. Python & Numpy
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 2. PyTorch CPU
    torch.manual_seed(seed)
    
    # 3. Force Deterministic Algorithms
    # This prevents non-deterministic CPU operations (like some views/reductions)
    torch.use_deterministic_algorithms(True)
    
    # 4. Limit CPU Threads
    # Setting this to 1 ensures operations are done in a fixed order.
    torch.set_num_threads(1)

In [3]:
def clip_local_energies(eloc, threshold=5.0):
    # Convert to numpy if it's a torch tensor, or vice versa
    eloc_real = np.real(eloc)
    median = np.median(eloc_real)
    mad = np.median(np.abs(eloc_real - median))
    
    # Standard safety check to avoid division by zero if MAD is 0
    if mad == 0:
        return eloc
        
    lower_bound = median - threshold * mad
    upper_bound = median + threshold * mad
    
    # Clip the values (keeping the imaginary part if it exists)
    # We create a copy to avoid modifying the original array in place
    clipped = np.clip(eloc_real, lower_bound, upper_bound)
    
    # If the original was complex, restore the imaginary part
    if np.iscomplexobj(eloc):
        return clipped + 1j * np.imag(eloc)
    return clipped 

def define_load_test(wf, numsamples,path_to_weights, Ee, clipped_e = False):
    test_samples_before = wf.sample(numsamples)
    print(f'The number of samples is {len(test_samples_before)}')
    # --- PART A: Check performance BEFORE loading (Baseline) ---
    wf.model.eval() 
    with torch.no_grad():
        test_gs_before = Ising2D_local_energies(Jz, Bx, Nx, Ny, test_samples_before, wf)
        gs_mean_b = np.round(np.mean(test_gs_before),4)
        gs_var_b = np.round(np.var(test_gs_before),4)
    print(f'Before loading weights, the ground state energy mean and variance are:')
    print(f'Mean E = {gs_mean_b}, var E = {gs_var_b}')
    print('====================================================================')

     # --- PART B: Remap and Load the Weights ---
    state_dict = torch.load(path_to_weights, map_location=torch.device('cpu'))   
    new_state_dict = {}
    for key, value in state_dict.items():
        # Strip prefixes and rename keys to match current architecture
        new_key = key.replace('model.', '').replace('cell.', 'rnn.')
        new_state_dict[new_key] = value
    # This line loads the RE-MAPPED weights
    wf.model.load_state_dict(new_state_dict, strict=False)
    print("Successfully remapped and loaded weights.")
    
    # --- PART C: Check performance AFTER loading ---
    with torch.no_grad():
        test_samples_after = wf.sample(numsamples)
        if clipped_e:
            # 1. Get raw energies
            raw_gs_after = Ising2D_local_energies(Jz, Bx, Nx, Ny, test_samples_after, wf)
    
            # 2. APPLY CLIPPING
            test_gs_after = clip_local_energies(raw_gs_after, threshold=5.0)
    
            # 3. Calculate statistics on cleaned data
            gs_mean_a = np.round(np.mean(test_gs_after), 4)
            gs_var_a = np.round(np.var(test_gs_after), 4)
    
            # Optional: Count how many were clipped to see if the model is unstable
            num_clipped = np.sum(np.real(raw_gs_after) != np.real(test_gs_after))
            print(f"Clipped {num_clipped} outlier samples out of {numsamples}")
        else:
            test_gs_after = Ising2D_local_energies(Jz, Bx, Nx, Ny, test_samples_after, wf)
            gs_mean_a = np.round(np.mean(test_gs_after),4)
            gs_var_a = np.round(np.var(test_gs_after),4)
    
    #wf.model.summary()
    #print('====================================================================')
    print(f'After loading weights, the ground state energy mean and variance are:')
    print(f'Mean E = {gs_mean_a}, var E = {gs_var_a}')
    print(f'DMRG energy (not exact in 2D) is {np.round(Ee,4)}')

In [4]:
Nx=10
Ny=10
Bx=3.0
units =70
Jz=np.ones((Nx,Ny))
nsamples = 10000
E_dmrg = -316.97705
seed=111
set_cpu_deterministic(seed)
fname = f'../2DTFIM_1DNQS_results_seed_{seed}/(10,10)'

## RNN variants

In [7]:
wf = RNNwavefunction(Nx, Ny, 'EuclRNN', units, seed=seed)

total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")
print('----------------------------------------------------------')
lrnn_w =f'{fname}/Euclidean/EuclRNN_Nx=10_70_10x10_ns=80_rmax=None_checkpoint.pt'
t0=time.time()
define_load_test(wf, nsamples,lrnn_w, Ee=E_dmrg)
t1=time.time()
print(f'Time taken ={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 5,252
----------------------------------------------------------
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = -300.2774, var E = 177.0887
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = -308.3666, var E = 69.9387
DMRG energy (not exact in 2D) is -316.977
Time taken =0.043 hrs


In [8]:
r_max=0.65
wf= RNNwavefunction_hyp(Nx, Ny, cell_type='HypRNN',r_max=r_max, bias_geom='hyp',
                           hyp_non_lin='id', units=units, seed=111)

total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")
print('----------------------------------------------------------')
lrnn_w =f'{fname}/(10,10)/PoincareRNN/HypRNN_Nx=10_70_10x10_ns=80_rmax=0.65_checkpoint.pt'
t0=time.time()
define_load_test(wf, nsamples,lrnn_w, Ee=E_dmrg)
t1=time.time()
print(f'Time taken ={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 5,252
----------------------------------------------------------
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = -300.0802, var E = 179.3449
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = -314.8574, var E = 26.7133
DMRG energy (not exact in 2D) is -316.977
Time taken =0.126 hrs


In [6]:
sc=6.0
wf= Lorentzwavefunction(systemsize_x=Nx, systemsize_y=Ny, cell_type='LorentzRNN', 
                        units=units, spatial_clamp=sc, seed=111)

total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")
print('----------------------------------------------------------')
lrnn_w =f'{fname}/LorentzRNN/LorentzRNN_70_10x10_ns=80_spatial_cl={sc}_checkpoint.pt'
t0=time.time()
define_load_test(wf, nsamples,lrnn_w, Ee=E_dmrg)
t1=time.time()
print(f'Time taken ={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 5,253
----------------------------------------------------------
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = -300.1534, var E = 178.4116
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = -315.162, var E = 17.0038
DMRG energy (not exact in 2D) is -316.977
Time taken =0.568 hrs


## GRU variants

In [11]:
wf = RNNwavefunction(Nx, Ny, 'EuclGRU', units, seed=seed)

total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")
print('----------------------------------------------------------')
lrnn_w =f'{fname}/Euclidean/EuclGRU_Nx=10_70_10x10_ns=80_rmax=None_checkpoint.pt'
t0=time.time()
define_load_test(wf, nsamples,lrnn_w, Ee=E_dmrg)
t1=time.time()
print(f'Time taken ={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,472
----------------------------------------------------------
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = -301.9273, var E = 161.443
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = -313.8298, var E = 37.5081
DMRG energy (not exact in 2D) is -316.977
Time taken =0.092 hrs


In [18]:
r_max=1.0
wf= RNNwavefunction_hyp(Nx, Ny, cell_type='HypGRU',r_max=r_max, bias_geom='hyp',
                           hyp_non_lin='id', units=units, seed=111)

total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")
print('----------------------------------------------------------')
lrnn_w =f'{fname}/PoincareGRU/HypGRU_Nx=10_70_10x10_ns=80_rmax=1.0_checkpoint.pt'
t0=time.time()
define_load_test(wf, nsamples,lrnn_w, Ee=E_dmrg)
t1=time.time()
print(f'Time taken ={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,472
----------------------------------------------------------
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = -298.7336, var E = 193.3624
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = -315.0972, var E = 25.7061
DMRG energy (not exact in 2D) is -316.977
Time taken =0.419 hrs


In [19]:
sc=4.0
wf= Lorentzwavefunction(systemsize_x=Nx, systemsize_y=Ny, cell_type='LorentzGRU', 
                        units=units, spatial_clamp=sc, seed=111)

total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")
print('----------------------------------------------------------')
lrnn_w =f'{fname}/LorentzGRU/10x10_LorentzGRU_Nx=10_70_ns=80_spatial_cl=4.0_checkpoint.pt'
t0=time.time()
define_load_test(wf, nsamples,lrnn_w, Ee=E_dmrg)
t1=time.time()
print(f'Time taken ={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,473
----------------------------------------------------------
The number of samples is 10000
Before loading weights, the ground state energy mean and variance are:
Mean E = -299.4462, var E = 181.9278
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = -314.6966, var E = 28.7928
DMRG energy (not exact in 2D) is -316.977
Time taken =0.951 hrs
